In [1]:
import os
import requests
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_fixed

# 1. Securely load the API key from the .env file
load_dotenv(override=True)
API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

# 2. Build the Resilient Extraction Engine
# This decorator tells Python: "If this function fails, wait 5 seconds and try again. Stop after 3 attempts."
@retry(stop=stop_after_attempt(3), wait=wait_fixed(5))
def fetch_stock_data(symbol):
    print(f"Attempting to fetch live market data for {symbol}...")
    
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey": API_KEY
    }
    
    # We enforce a 10-second timeout. If the server hangs, we cut the connection and trigger a retry.
    response = requests.get(url, params=params, timeout=10)
    
    # This automatically raises an error if the server returns a 400 or 500 level crash code
    response.raise_for_status() 
    
    data = response.json()
    
    # Alpha Vantage returns a 200 OK even when you hit a rate limit. We must catch that manually.
    if "Information" in data or "Error Message" in data:
        raise ValueError(f"API Rate Limit or Error Hit: {data}")
        
    return data

# 3. Execute the Pipeline
if __name__ == "__main__":
    try:
        # We are pulling HDB (HDFC Bank's US-listed ADR) to ensure reliable data flow
        raw_market_data = fetch_stock_data("HDB")
        print("✅ Success! Data extracted securely.")
        
        # Navigate the JSON dictionary to isolate just the time series data
        time_series = raw_market_data.get("Time Series (Daily)", {})
        
        # Grab the most recent trading day
        latest_date = list(time_series.keys())[0]
        latest_metrics = time_series[latest_date]
        
        print(f"\nLatest Trading Metrics for {latest_date}:")
        print(f"Open: {latest_metrics['1. open']}")
        print(f"Close: {latest_metrics['4. close']}")
        print(f"Volume: {latest_metrics['5. volume']}")
        
    except Exception as e:
        print(f"❌ Pipeline failed after 3 automated retries. Final Error: {e}")

Attempting to fetch live market data for HDB...
✅ Success! Data extracted securely.

Latest Trading Metrics for 2026-09-25:
Open: 23.2700
Close: 23.0100
Volume: 4885447


In [2]:
# ==========================================
# PHASE 2: DATA TRANSFORMATION & SQL UPSERT
# ==========================================
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert

def transform_to_dataframe(raw_data, symbol):
    """Converts the nested JSON into a clean Pandas DataFrame."""
    time_series = raw_data.get("Time Series (Daily)", {})
    
    # Convert JSON dictionary to DataFrame and transpose it
    df = pd.DataFrame.from_dict(time_series, orient='index')
    
    # Rename Alpha Vantage's columns to match our SQL database schema
    df = df.rename(columns={
        "1. open": "open_price",
        "2. high": "high_price",
        "3. low": "low_price",
        "4. close": "close_price",
        "5. volume": "volume"
    })
    
    # Clean up the index and set standard data types
    df.index.name = "trade_date"
    df = df.reset_index()
    df["symbol"] = symbol
    df["trade_date"] = pd.to_datetime(df["trade_date"])
    
    # Convert numeric columns from strings to floats/ints
    cols_to_float = ["open_price", "high_price", "low_price", "close_price"]
    df[cols_to_float] = df[cols_to_float].astype(float)
    df["volume"] = df["volume"].astype(int)
    
    return df

def upsert_to_postgres(df):
    """Loads data into PostgreSQL, automatically overriding duplicates."""
    # We use the os library (imported in Phase 1) to grab the DB string
    engine = create_engine(os.getenv("DATABASE_URL"))
    metadata = MetaData()
    
    # Reflect the existing table you just created in pgAdmin4
    market_table = Table("market_data", metadata, autoload_with=engine)
    
    # Convert DataFrame to a list of dictionaries for SQLAlchemy
    records = df.to_dict(orient="records")
    
    with engine.begin() as conn:
        for record in records:
            # 1. Build the base Insert statement
            stmt = insert(market_table).values(**record)
            
            # 2. Build the "On Conflict" (UPSERT) logic targeting the excluded (new) data
            upsert_stmt = stmt.on_conflict_do_update(
                index_elements=["trade_date", "symbol"],
                set_={
                    "open_price": stmt.excluded.open_price,
                    "high_price": stmt.excluded.high_price,
                    "low_price": stmt.excluded.low_price,
                    "close_price": stmt.excluded.close_price,
                    "volume": stmt.excluded.volume
                }
            )
            
            # 3. Execute the database transaction
            conn.execute(upsert_stmt)

# ==========================================
# FINAL EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    try:
        symbol = "HDB"
        
        # 1. Extract
        raw_market_data = fetch_stock_data(symbol)
        print("✅ Data Extracted from Alpha Vantage.")
        
        # 2. Transform
        df = transform_to_dataframe(raw_market_data, symbol)
        print(f"✅ Data Transformed. Processed {len(df)} days of trading data.")
        
        # 3. Load
        upsert_to_postgres(df)
        print("✅ Data successfully UPSERTED into PostgreSQL.")
        
    except Exception as e:
        print(f"❌ Pipeline failed: {e}")

Attempting to fetch live market data for HDB...
✅ Data Extracted from Alpha Vantage.
✅ Data Transformed. Processed 100 days of trading data.
✅ Data successfully UPSERTED into PostgreSQL.
